# 03 — Factor covariance F(t)

**Step 5 of the model.** One `K×K` covariance of factor returns per forecast
date, on a rolling 60-month window:

$$\Sigma(t) = X(t)\,F(t)\,X(t)^\top + \Delta(t)$$

`F(t)` uses factor returns $f(s)$ for $s \le t$ and forecasts the covariance of
$f(t+1)$. That convention is used everywhere downstream: a bias test pairs `F(t)`
with `f(t+1)`, and a portfolio held at the end of month *t* is priced with
`X(t)`, `F(t)`, `Δ(t)`.

Reads `ryan_sample_factor_returns`. Writes `ryan_sample_factor_covariance`
(long: `forecast_date, factor_i, factor_j, cov` plus every intermediate stage)
and `ryan_sample_factor_cov_diag`. Run top to bottom.

### The four stages, all stored

| stage | what it adds | why |
|---|---|---|
| `raw` | equal-weighted sample covariance on the window | the baseline everything is compared with |
| `nw` | exponential decay; USE4's split of vol and correlation half-lives | risk levels move; 153 correlations need more data than 18 vols |
| `eigen` | eigenfactor adjustment | at T/K = 3.3 the smallest eigenvalues are understated by tens of percent |
| `final` | volatility regime adjustment (VRA) | rescales the whole matrix to the recent out-of-sample bias |

**Newey-West is in the code and off by default** (`FACTOR_NW_LAGS = 0`). Stage 2
measures why. It was in the plan and it makes `F` worse here.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import factor_cov
import riskmath
import biastest

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Same validated palette as 02. Low-contrast hues always carry a direct label.
STYLE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, ACCENT, MUTED = "#2a78d6", "#e34948", "#8a8a83"
STAGE_COLORS = {"raw": MUTED, "nw": "#eda100", "eigen": "#1baf7a", "final": INK}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


# Rebuilding is ~10 s with 1000 eigen simulations; False resumes from the table.
REBUILD_COV = True

print(f"window {config.COV_WINDOW} months, vol/corr half-life "
      f"{config.FACTOR_VOL_HALFLIFE}/{config.FACTOR_CORR_HALFLIFE}, NW lags {config.FACTOR_NW_LAGS}")
print(f"eigen sims {config.EIGEN_SIMS}, scale {config.EIGEN_SCALE}; VRA half-life {config.VRA_HALFLIFE}")

In [ ]:
engine = db.get_engine()
db.check_connection(engine)

---
## Stage 1 — Load f(t) and confirm it is the model 02 built

The anchor numbers are from the handover. If any of them moves, something
upstream changed. Style vols of 17–23%/yr would mean the exposure lag broke.

In [ ]:
if not db.table_exists(engine, config.TBL_FACTOR_RETURNS):
    raise RuntimeError(f"{config.TBL_FACTOR_RETURNS} is missing — run 02_factor_returns.ipynb first.")

fl = db.read_table(config.TBL_FACTOR_RETURNS, engine)
fl["eom"] = pd.to_datetime(fl["eom"])
FR = fl.pivot(index="eom", columns="factor", values="factor_return")
FR = FR[factor_cov.canonical_order(FR.columns)]
styles = list(config.STYLE_DESCRIPTORS)
inds = [c for c in FR.columns if c.startswith("ind_")]

ann = pd.DataFrame({"mean %/yr": FR.mean() * 1200, "vol %/yr": FR.std() * np.sqrt(12) * 100})
anchors = {"market": (7.87, 15.38), "value": (2.21, 4.67), "momentum": (2.12, 6.60), "size": (1.84, 4.96)}
print(f"T = {len(FR)} months × K = {FR.shape[1]} factors, {FR.index[0].date()} → {FR.index[-1].date()}")
print(f"{'✅' if not FR.isna().any().any() else '❌'} complete panel (no factor missing in any month)")
for f, (m, v) in anchors.items():
    ok = abs(ann.loc[f, "mean %/yr"] - m) < 0.05 and abs(ann.loc[f, "vol %/yr"] - v) < 0.05
    print(f"{'✅' if ok else '❌'} {f:<9s} {ann.loc[f, 'mean %/yr']:6.2f}%/yr at {ann.loc[f, 'vol %/yr']:5.2f}% vol"
          f"   (handover: {m} / {v})")
sv = ann.loc[styles, "vol %/yr"]
print(f"{'✅' if sv.max() < 10 else '❌'} style vols {sv.min():.1f}–{sv.max():.1f}%/yr (anchor 4–8)")

---
## Stage 2 — What each component is worth, measured out of sample

Every row below builds `F(t)` at all 181 dates. It is scored only on the
**following** month, which the estimator never saw. The eigenfactor adjustment
is switched off here so each row isolates one component. Columns:

- **factor bias**: std of $f_k(t+1)/\sigma_k(t)$, averaged over the 18 factors.
- **smallest-eigvec bias**: the same for the portfolio `F(t)` believes is
  safest. This is what a min-variance optimizer buys.
- **min-var ptf bias**: the minimum-variance factor portfolio with unit market
  exposure, rebuilt each month.

One is calibrated; above one means risk was understated. The 95% band over 180
dates is roughly 1 ± 0.10.

In [ ]:
def score(path, stage="final"):
    ez = biastest.eigen_z(path, FR, stage, basis_stage=stage)
    return {
        "factor bias": biastest.factor_z(path, FR, stage).std(ddof=1).mean(),
        "smallest-eigvec bias": riskmath.bias_stat(ez[:, 0]),
        "min-var ptf bias": riskmath.bias_stat(biastest.minvar_factor_z(path, FR, stage)),
    }


variants = {
    "equal weight, no NW (= raw)": dict(vol_half_life=float("inf"), corr_half_life=float("inf"), lags=0),
    "decay 24/48, no NW": dict(lags=0),
    "decay 24/48, NW lags 1": dict(lags=1),
    "decay 24/48, NW lags 2": dict(lags=2),
    "decay 24/48, NW lags 3 (rule of thumb)": dict(lags=None),
}
comp = pd.DataFrame({
    name: score(factor_cov.estimate(FR, eigen_sims=0, vra_min_obs=10**6, verbose=False, **kw))
    for name, kw in variants.items()
}).T
display(comp.round(3))
print("95% band for a calibrated forecast over 180 dates:",
      "[{:.3f}, {:.3f}]".format(*riskmath.bias_ci(180)))

**Decay is harmless and Newey-West is not.** Each added lag makes every column
worse, and the damage concentrates in exactly the directions an optimizer uses.

The mechanism: each lag term adds $\Gamma_l$, an 18×18 cross-autocovariance
estimated from ~57 months. The factors' true lag-1 autocorrelations are small
(next cell), so $\Gamma_l$ is almost pure noise. Noise added to a covariance
spreads its spectrum and drags the smallest eigenvalue down. Bartlett weights
guarantee the result is PSD. They do not guarantee it is right.

USE4 can afford NW because it estimates on **daily** returns, with thousands of
observations per window. On 60 monthly observations the autocorrelation signal
isn't there to find. `FACTOR_NW_LAGS = 0`, and this table re-runs on every
execution, so if more history ever changes the verdict, it shows here.

In [ ]:
ac = FR.apply(lambda s: s.autocorr(1))
print("lag-1 autocorrelation of factor returns (full sample):")
print(f"   median |ρ| {ac.abs().median():.3f}, max |ρ| {ac.abs().max():.3f} ({ac.abs().idxmax()})")
print(f"   2/√T noise band ±{2 / np.sqrt(len(FR)):.3f} — "
      f"{(ac.abs() > 2 / np.sqrt(len(FR))).sum()} of {len(ac)} factors outside it")

---
## Stage 3 — Estimate F(t)

The resume pattern from `run_pipeline`: rebuild when asked or when the table is
missing, otherwise load. Everything below runs on the same `path` object
either way.

In [ ]:
if REBUILD_COV or not db.table_exists(engine, config.TBL_FACTOR_COV):
    path = factor_cov.estimate(FR)
    rebuilt = True
else:
    path = factor_cov.path_from_long(
        db.read_table(config.TBL_FACTOR_COV, engine),
        db.read_table(config.TBL_FACTOR_COV_DIAG, engine),
    )
    rebuilt = False
    print(f"📂 loaded F(t) at {len(path.dates)} forecast dates from {config.TBL_FACTOR_COV}")

diag = path.diag
print(f"\nforecast dates   {path.dates[0].date()} → {path.dates[-1].date()}  ({len(path.dates)}, "
      f"{len(path.dates) - 1} with a next month to test on)")

### Verify before believing

In [ ]:
checks = {
    "181 forecast dates (240 months, 60-month window)": len(path.dates) == len(FR) - config.COV_WINDOW + 1,
    "first forecast date is the 60th month": path.dates[0] == FR.index[config.COV_WINDOW - 1],
    "factor order is canonical (market, industries, styles)": path.factors == list(FR.columns),
    "every stage symmetric & PSD at every date": all(
        riskmath.is_psd(m) for s in factor_cov.STAGES for m in path.stages[s]),
    "final F strictly positive definite everywhere": all(
        np.linalg.eigvalsh(m).min() > 0 for m in path.stages["final"]),
    "industry-style covariances are populated (one joint F)": all(
        np.abs(path.at(d).loc[inds, styles].to_numpy()).max() > 0 for d in path.dates[::30]),
}
for k, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {k}")

display(diag.drop(columns=["forecast_date"]).describe().T[["mean", "min", "50%", "max"]].round(3))

---
## Stage 4 — The eigenfactor adjustment, visibly

**Left:** the simulated bias λ(k) by eigenvalue rank at every date. The
smallest eigenvalue's variance is understated by roughly half, and the largest
is about right. **Right:** out of sample, the bias statistic of each
eigenportfolio under each stage. Each rank is the same portfolio across stages,
because they all use the pre-adjustment eigenbasis. The adjustment should pull
the left end of the curve down toward one.

In [ ]:
lam = np.array([riskmath.eigenfactor_adjust(
    path.stages["nw"][i], riskmath.exp_weights(config.COV_WINDOW, config.FACTOR_CORR_HALFLIFE),
    n_sims=100, scale=config.EIGEN_SCALE, seed=i)[1]["lam"] for i in range(0, len(path.dates), 10)])

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
rank = np.arange(1, lam.shape[1] + 1)
ax[0].fill_between(rank, lam.min(0), lam.max(0), color=INK, alpha=.15, lw=0)
ax[0].plot(rank, np.median(lam, 0), color=INK, lw=2, marker="o", ms=4)
ax[0].axhline(1, color=MUTED, lw=1)
ax[0].set(title="Simulated eigenvalue bias λ(k) — median and range over dates",
          xlabel="eigenvalue rank (1 = smallest)", ylabel="λ: true / estimated vol")
ci = riskmath.bias_ci(len(path.dates) - 1)
ax[1].axhspan(*ci, color=MUTED, alpha=.15, lw=0)
for s in factor_cov.STAGES:
    ez = biastest.eigen_z(path, FR, s, basis_stage="nw")
    b = ez.std(axis=0, ddof=1)
    ax[1].plot(rank, b, color=STAGE_COLORS[s], lw=2, marker="o", ms=4, label=s)
    ax[1].annotate(f" {s}", (rank[0], b[0]), color=STAGE_COLORS[s], fontsize=9, ha="right",
                   xytext=(-6, 0), textcoords="offset points", va="center")
ax[1].axhline(1, color=MUTED, lw=1)
ax[1].set(title="Out-of-sample eigenportfolio bias, by stage", xlabel="eigenvalue rank (1 = smallest)",
          ylabel="bias statistic", xlim=(-1.5, rank[-1] + .5))
ax[1].legend(frameon=False, ncol=4, fontsize=9)
for a in ax:
    tidy(a)
plt.tight_layout()
plt.show()

In [ ]:
summary = biastest.summarize(path, FR)
display(summary.round(3))
print(f"95% band: [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"\nmin-variance portfolio bias: raw {summary.loc['raw', 'min-var ptf bias']:.2f} → "
      f"final {summary.loc['final', 'min-var ptf bias']:.2f}")
print("The min-var row is the one that matters for 07. It is the portfolio an optimizer")
print("actually buys, and without the adjustment its risk is understated by the margin above.")

**Reading the table.** The eigenfactor adjustment does the heavy lifting: min-var
bias goes from 1.63 to about 1.17 and the smallest eigenportfolio from about 2.0
to 1.3. It slightly **over**states the average factor (0.955), because
rescaling the spectrum raises every direction a little.

VRA then does its own job, which is the overall *level*. It pulls factor bias to
~1.0 by shrinking the matrix (λ mostly < 1). Shrinking the matrix also shrinks
the min-var direction, which gives back part of the eigen gain (≈1.17 → 1.25).
The two corrections disagree about which portfolios matter, and that follows from
design: VRA is calibrated on the average factor, not on the tail of the
spectrum. The residual ~25% here is for the pure *factor* min-var portfolio, the
most extreme case. On real asset portfolios, where specific risk dilutes it, 05
measures 1.07 for unconstrained long-short min-var and 1.04 for long-only.

---
## Stage 5 — Volatility forecasts through time

The forecast vol of each style factor against its realized absolute return.
The forecast should rise into 2020 and 2022 and fall afterwards. VRA (lower
panel) scales the whole matrix when recent months have run hotter than
forecast. It stays at 1 for the first 12 forecast dates because it needs 12
out-of-sample observations before it acts.

In [ ]:
vol = pd.DataFrame(np.sqrt(np.diagonal(path.stages["final"], axis1=1, axis2=2) * 12) * 100,
                   index=path.dates, columns=path.factors)

fig, axes = plt.subplots(2, 1, figsize=(13, 6.4), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
ax = axes[0]
series = dict(zip(styles + ["market"], STYLE_COLORS + [ACCENT]))
for name, color in series.items():
    ax.plot(vol.index, vol[name], color=color, lw=1.8)
# Direct labels nudged apart in log space, as in 02, so names ending close
# together stay legible -- the label is what carries identity for the paler hues.
ends = np.log(vol[list(series)].iloc[-1]).sort_values()
gap, last, placed = 0.045, -np.inf, {}
for name in ends.index:
    placed[name] = max(ends[name], last + gap)
    last = placed[name]
for name, color in series.items():
    ax.annotate(f" {name}", (vol.index[-1], np.exp(placed[name])), color=color, fontsize=9, va="center")
ax.set(title="Forecast factor volatility, F(t) final stage", ylabel="annualized vol, %", yscale="log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}" if v in (3, 5, 7, 15) else ""))
ax.margins(x=.08)
axes[1].plot(diag["forecast_date"], diag["vra_lambda"], color=INK, lw=1.5)
axes[1].axhline(1, color=MUTED, lw=1)
axes[1].set(title="Volatility regime adjustment λ_F (multiplies vols)", ylabel="λ_F")
for a in axes:
    tidy(a)
plt.tight_layout()
plt.show()

In [ ]:
# Realized-vs-forecast for the market factor, month by month: |f(t+1)| against
# σ(t). The 12-month rolling bias statistic is what MRAD averages over.
fz = biastest.factor_z(path, FR, "final")
roll = fz.apply(lambda s: biastest.rolling_bias(s.to_numpy()).set_axis(s.index))
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.axhspan(*riskmath.bias_ci(12), color=MUTED, alpha=.15, lw=0)
ax.plot(roll.index, roll["market"], color=ACCENT, lw=1.6, label="market")
ax.plot(roll.index, roll[styles].median(axis=1), color=INK, lw=1.6, label="median style")
ax.axhline(1, color=MUTED, lw=1)
ax.set(title="12-month rolling bias statistic (shaded: 95% band for a 12-month window)",
       ylabel="bias statistic")
ax.legend(frameon=False, fontsize=9)
tidy(ax)
plt.tight_layout()
plt.show()
print(f"MRAD, market {riskmath.mrad(roll['market'].dropna()):.3f}; "
      f"median over styles {np.median([riskmath.mrad(roll[s].dropna()) for s in styles]):.3f}")

---
## Stage 6 — The latest F

The matrix 06–08 price portfolios with. Correlations shown with every cell
labelled.

In [ ]:
latest = path.at(path.dates[-1])
core = ["market"] + styles
sd = np.sqrt(np.diag(latest))
corr = latest / np.outer(sd, sd)

fig, ax = plt.subplots(figsize=(7, 5.6))
c = corr.loc[core, core]
im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(core)), core, rotation=45, ha="right")
ax.set_yticks(range(len(core)), core)
for i in range(len(core)):
    for j in range(len(core)):
        v = c.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > .55 else "#1a1a19")
plt.colorbar(im, shrink=.8, label="correlation")
ax.set_title(f"Forecast correlations at {path.dates[-1].date()}")
plt.tight_layout()
plt.show()

vols_now = pd.Series(sd * np.sqrt(12) * 100, index=latest.index).round(2)
print("forecast vols at the latest date, %/yr:")
display(vols_now.to_frame("vol %/yr").T)

---
## Stage 7 — Write the artifacts

Long format, both triangles, every stage in its own column. `cov` is the final
stage and is the only column 05–08 price with by default. The others are there
so a bias test can ask what a stage bought without recomputing anything.

In [ ]:
if rebuilt:
    long = factor_cov.to_long(path)
    db.write_table(long, config.TBL_FACTOR_COV, engine,
                   unique_on=["forecast_date", "factor_i", "factor_j"])
    db.write_table(diag, config.TBL_FACTOR_COV_DIAG, engine, unique_on=["forecast_date"])
    print(f"{len(path.dates)} dates × {len(path.factors)}² = {len(long):,} rows")
else:
    print("resumed from the table — nothing written")

---
## What 04 does next

`Δ(t)`, the specific half of Σ, at the same 181 forecast dates. 04 reads the
forecast dates from this table, so the two halves always exist together.
Rolling, decay-weighted variance of each asset's own `u`, Bayesian shrinkage
toward its size-decile mean, then a specific VRA.